In [15]:
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin

import torch
import torchvision.models as models
from torchvision import transforms

import time

# Chargement du modèle pré-entrainé
model = models.squeezenet1_0(pretrained=True)
model = model.float().eval()

# Pré-traitement des données
preprocess = transforms.Compose([
    # transforms.Resize(64),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Encapsulation dans une classe au format Scikit-Learn
class CNNFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, model, preprocess):
        self.model = model
        self.preprocess = preprocess

    def transform(self, X):
        img_tensor = self.preprocess(X)
        with torch.no_grad():
            features = self.model(img_tensor)
        return features

    def fit(self, X, y=None):
        return self

# Exemple sur des données simulées
n = 4
h, w = 64, 64
images = np.random.randn(n, h, w, 3)
images = torch.from_numpy(images).permute(0, 3, 1, 2) # -> (n, 3, h, w)
images = images.float() # float64 --> float32

cnn_model = CNNFeatureExtractor(model, preprocess)
start_time = time.time()
features = cnn_model.transform(images)
end_time = time.time()

print('Représentations de taille {} calculées en {:.2f}s.'.format(features.shape, end_time - start_time))

Représentations de taille torch.Size([4, 1000]) calculées en 0.01 s.
